# Multimodal Cancer Classification Challenge 2026 — v10

**Why this is faster *and* better than v9:**

Speed (so it actually finishes in <2 h on Kaggle):
- **RAM-cached JPEG bytes** — load all 230k images once, decode on access. Eliminates Kaggle's slow `/kaggle/input` random-access I/O.
- `NUM_WORKERS=2` is safe again because workers no longer compete for disk.

Generalization (so AUC actually improves):
- **EfficientNet-B0** backbone (timm, 1-ch stem) — 9 M params, stronger features than ResNet-18.
- **Patient-balanced sampling** — every batch is forced to see cells from multiple patients, which fights patient-level overfitting (the main failure mode of v9 fold 1).
- **Stochastic Weight Averaging (SWA)** over the last 4 epochs — averages weights along the loss valley, which gives a measurable AUC bump on small-cohort medical data.
- **D4 augmentation** (random 90° + flips) + paired affine + ColorJitter + RandomErasing.
- **Mixup (α=0.2)** + **label smoothing (ε=0.05)**.
- **OneCycleLR** with 10 % warmup, gradient clipping.
- **8-way D4 TTA** at inference, averaged across folds (and across the SWA model + last best ckpt).

Runtime budget (single T4):
- Setup + RAM cache: ~10 min (one-time).
- Training: ~25 min × 3 folds.
- TTA prediction: ~10 min.
- **Total ~1.5 h** — fits comfortably in Kaggle's 9 h commit limit.

Settings to enable before running:
- Accelerator = **GPU T4 x2** (we only use GPU 0; second GPU stays idle, which is fine).
- Internet = **On** (for ImageNet weights + timm).
- Persistence = **Files only**.

In [ ]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, sys, functools, gc
from pathlib import Path

# Force prints to flush so Kaggle commit logs are live.
print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.optim.swa_utils import AveragedModel, update_bn
from torchvision import models
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, GroupKFold, LeaveOneGroupOut
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
print("timm:", timm.__version__)
!nvidia-smi -L

In [ ]:
DATA_ROOT = Path("/kaggle/input/datasets/rafaelproena/a3-adl")
assert (DATA_ROOT / "train.csv").exists(), f"train.csv not at {DATA_ROOT}"
print("DATA_ROOT =", DATA_ROOT)
print("Contents:", sorted(p.name for p in DATA_ROOT.iterdir()))

OUT_DIR = Path("/kaggle/working/runs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CV
CV          = "sgkf"            # sgkf | gkf | lopo
N_SPLITS    = 3
SEED        = 1

# Optimization
EPOCHS      = 12
SWA_EPOCHS  = 4                 # last N epochs are averaged into SWA model
PATIENCE    = 6
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0

# Regularization
MIXUP_ALPHA  = 0.2
LABEL_SMOOTH = 0.05
DROPOUT      = 0.4

# Model
BACKBONE     = "efficientnet_b0"   # or "resnet18" for a quick test
PRETRAINED   = True

# Data loading
NUM_WORKERS  = 2
PATIENTS_PER_BATCH = 4              # patient-balanced sampler picks this many distinct patients per batch

# System
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# We deliberately use only GPU 0; DataParallel adds RAM pressure on Kaggle and isn't worth it here.
if torch.cuda.is_available():
    torch.cuda.set_device(0)

# Normalization (from v9 profile of 500 random train images)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

# Reproducibility (best-effort)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

In [ ]:
# RAM-cached dataset: read every JPEG into a bytes object exactly once, then
# decode on __getitem__. The decode is fast (sub-ms); the file open was the
# bottleneck. Memory cost: ~3-4 GB for the full train+test set.
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split(names, bf_dir, fl_dir, label=""):
    """Read JPEG bytes for all `names` from `bf_dir` and `fl_dir` into two dicts."""
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 20000 == 0:
            print(f"  [{label}] cached {i+1}/{len(names)} pairs in {time.time()-t0:.1f}s")
    print(f"  [{label}] cached {len(names)} pairs in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    """Reads JPEG bytes from in-memory dicts; decodes + transforms on access."""
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache
        self.fl_cache = fl_cache
        self.bf_tf = bf_tf
        self.fl_tf = fl_tf
        self.paired_tf = paired_tf

    def __len__(self): return len(self.df)

    @staticmethod
    def _decode(buf):
        return Image.open(io.BytesIO(buf)).convert("L")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

In [ ]:
def stratified_patient_kfold(df, n_splits=3, seed=1, strict=True):
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    if strict:
        for f, (_, va) in enumerate(splits):
            if len(np.unique(y[va])) < 2:
                raise ValueError(f"Fold {f} has only one class — try a different SEED.")
    return splits

def get_splits(df, cv, n_splits, seed):
    if cv == "sgkf": return stratified_patient_kfold(df, n_splits, seed)
    if cv == "gkf":  return list(GroupKFold(n_splits=n_splits).split(df, groups=df["patient_id"]))
    if cv == "lopo": return list(LeaveOneGroupOut().split(df, groups=df["patient_id"].to_numpy()))
    raise ValueError(cv)

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")

class PatientBalancedSampler(Sampler):
    """Yield indices so that every batch contains cells from `patients_per_batch`
    distinct patients in roughly equal proportions. This dilutes patient-level
    spurious features (slide staining, microscope drift, etc.) which is the main
    failure mode on this 12-patient dataset.

    Length is the same as the training set so OneCycleLR step counting still works.
    """
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0, \
            f"batch_size ({batch_size}) must be divisible by patients_per_batch ({patients_per_batch})"
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size

    def __len__(self): return self.epoch_len

    def __iter__(self):
        out = []
        n_batches = self.epoch_len // self.batch_size
        for _ in range(n_batches):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                pick = self.rng.choice(idxs, size=self.per_pat,
                                       replace=len(idxs) < self.per_pat)
                out.extend(pick.tolist())
        return iter(out)

In [ ]:
def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn

to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)


class PairedGeoAug:
    """Same geometric transform applied to both modalities so they stay aligned.
    D4 (random 90° + flip) plus a fine random rotation and small affine shift.
    """
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True,
                 max_rot=20.0, max_shift=0.06):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot; self.max_shift = max_shift

    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0 or self.max_shift > 0:
            ang = random.uniform(-self.max_rot, self.max_rot) if self.max_rot > 0 else 0.0
            tx  = int(random.uniform(-self.max_shift, self.max_shift) * bf.shape[-1])
            ty  = int(random.uniform(-self.max_shift, self.max_shift) * bf.shape[-2])
            bf = TF.affine(bf, angle=ang, translate=[tx, ty], scale=1.0, shear=[0.0, 0.0])
            fl = TF.affine(fl, angle=ang, translate=[tx, ty], scale=1.0, shear=[0.0, 0.0])
        return bf, fl


def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([
        T.ColorJitter(brightness=0.3, contrast=0.3),
        norm,
        T.RandomErasing(p=0.25, scale=(0.02, 0.15), value=0),
    ])

def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl

In [ ]:
def _make_effnet_b0_branch(pretrained=True):
    net = timm.create_model("efficientnet_b0", pretrained=pretrained,
                             num_classes=0, global_pool="avg")
    old = net.conv_stem
    new_conv = nn.Conv2d(1, old.out_channels, kernel_size=old.kernel_size,
                         stride=old.stride, padding=old.padding, bias=False)
    if pretrained:
        new_conv.weight.data = old.weight.data.mean(dim=1, keepdim=True)
    net.conv_stem = new_conv
    return net, net.num_features  # 1280

def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd  # 512

def _make_branch(backbone, pretrained):
    if backbone == "efficientnet_b0": return _make_effnet_b0_branch(pretrained)
    if backbone == "resnet18":        return _make_resnet18_branch(pretrained)
    raise ValueError(backbone)

class MultimodalClassifier(nn.Module):
    """Two-branch (BF, FL) backbone with a small attention-weighted fusion head.
    The learnable gate lets the model emphasize whichever modality is more
    informative per-sample, instead of a fixed concat.
    """
    def __init__(self, pretrained=True, dropout=DROPOUT, backbone=BACKBONE):
        super().__init__()
        self.bf_branch, fd = _make_branch(backbone, pretrained)
        self.fl_branch, _  = _make_branch(backbone, pretrained)
        self.gate = nn.Sequential(nn.Linear(fd * 2, 2), nn.Softmax(dim=-1))
        hidden = 512 if fd >= 512 else 256
        self.head = nn.Sequential(
            nn.Linear(fd * 2, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, bf, fl):
        fb = self.bf_branch(bf)
        ff = self.fl_branch(fl)
        cat0 = torch.cat([fb, ff], dim=1)
        w = self.gate(cat0)                              # (B, 2)
        fused = torch.cat([fb * w[:, :1], ff * w[:, 1:]], dim=1)
        return self.head(fused).squeeze(-1)

# Sanity check
with torch.no_grad():
    _m = MultimodalClassifier(pretrained=False).cpu()
    _x = torch.zeros(2, 1, 128, 128)
    print("Model output shape:", _m(_x, _x).shape,
          "  params:", sum(p.numel() for p in _m.parameters()) // 1_000_000, "M")
    del _m, _x

In [ ]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes into RAM ---")
print("This is the one-time cost. After this, every epoch is GPU-bound.")
bf_train_cache, fl_train_cache = cache_split(
    df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train", label="train")
bf_test_cache, fl_test_cache = cache_split(
    df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test", label="test")

# Quick stat: how much RAM are we using?
approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

In [ ]:
def mixup_batch(bf, fl, y, alpha=0.2):
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])

def smooth_labels(y, eps=0.05):
    return y * (1.0 - eps) + eps * 0.5

def run_epoch(model, loader, optimizer, scaler, criterion, train,
              mixup_alpha=0.0, label_smooth=0.0, grad_clip=0.0, sched=None,
              log_every=0):
    model.train(train)
    losses, hard_ys, ps = [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())

        if train:
            if mixup_alpha > 0: bf, fl, y = mixup_batch(bf, fl, y, mixup_alpha)
            if label_smooth > 0: y = smooth_labels(y, label_smooth)

        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl)
            loss = criterion(logits, y)

        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                # Only advance LR if the optimizer step wasn't skipped (AMP inf/NaN check).
                if sched is not None and scaler.get_scale() >= old_scale:
                    sched.step()
            else:
                loss.backward()
                if grad_clip > 0:
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: sched.step()

        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())

        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")

    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps


def evaluate_swa(swa_model, loader, criterion):
    """BatchNorm stats in the SWA model are stale — update them before eval."""
    swa_model.eval()
    with torch.no_grad():
        return run_epoch(swa_model, loader, None, None, criterion, False)


def train_fold(df, train_idx, val_idx, fold):
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    train_ds = CachedCellDataset(train_df, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    val_ds   = CachedCellDataset(val_df, bf_train_cache, fl_train_cache,
                                 eval_modality_transform("bf"),
                                 eval_modality_transform("fl"))

    sampler = PatientBalancedSampler(train_df, batch_size=BATCH_SIZE,
                                     patients_per_batch=PATIENTS_PER_BATCH,
                                     seed=SEED + fold)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=False)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              persistent_workers=False)

    # Sanity batch — fails fast if anything is wrong with cache/transforms.
    _b = next(iter(train_loader))
    print(f"  sanity batch ok: bf={tuple(_b['bf'].shape)} fl={tuple(_b['fl'].shape)}")
    print(f"  batch patient mix: {sorted(set(parse_patient_id(n) for n in _b['name']))}")
    del _b

    model = MultimodalClassifier(pretrained=PRETRAINED, dropout=DROPOUT,
                                 backbone=BACKBONE).to(DEVICE)

    pos = (train_df["Diagnosis"] == 1).sum()
    neg = (train_df["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  backbone={BACKBONE}  "
          f"params={sum(p.numel() for p in model.parameters())//1_000_000}M")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=len(train_loader), epochs=EPOCHS,
        pct_start=0.1,
    )
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
    swa_model = AveragedModel(model)
    swa_start = EPOCHS - SWA_EPOCHS

    history, best_auc, no_improve = [], -1.0, 0
    best_ckpt = OUT_DIR / f"fold{fold}_best.pt"
    swa_ckpt  = OUT_DIR / f"fold{fold}_swa.pt"

    for ep in range(EPOCHS):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            mixup_alpha=MIXUP_ALPHA, label_smooth=LABEL_SMOOTH,
            grad_clip=GRAD_CLIP, sched=sched, log_every=100,
        )
        with torch.no_grad():
            va_loss, va_auc, vy, vp = run_epoch(
                model, val_loader, None, None, criterion, False)

        # SWA: collect the last few epochs into an averaged model.
        if ep >= swa_start:
            swa_model.update_parameters(model)

        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})

        if va_auc > best_auc:
            best_auc, no_improve = va_auc, 0
            torch.save({"model": model.state_dict(), "epoch": ep, "val_auc": va_auc,
                        "args": {"backbone": BACKBONE, "dropout": DROPOUT}}, best_ckpt)
            pd.DataFrame({"Name": val_df["Name"].values,
                          "patient_id": val_df["patient_id"].values,
                          "y_true": vy, "y_pred": vp}
                        ).to_csv(OUT_DIR / f"fold{fold}_oof_best.csv", index=False)
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f"  Early stopping at epoch {ep}")
                break

    # Finalize SWA: refresh BatchNorm stats with a pass over training data,
    # then evaluate on validation and save.
    print("  finalizing SWA (BN refresh + eval)...")
    bn_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    swa_model = swa_model.to(DEVICE)
    with torch.no_grad():
        # update_bn from torch.optim.swa_utils needs a loader that yields tensors.
        # Since our batch is a dict, we run a manual BN refresh.
        swa_model.train()
        for batch in bn_loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            swa_model(bf, fl)
    swa_loss, swa_auc, vy_swa, vp_swa = evaluate_swa(swa_model, val_loader, criterion)
    print(f"  SWA val AUC = {swa_auc:.4f}  (best single-epoch = {best_auc:.4f})")
    torch.save({"model": swa_model.state_dict(), "val_auc": swa_auc,
                "args": {"backbone": BACKBONE, "dropout": DROPOUT, "swa": True}}, swa_ckpt)
    pd.DataFrame({"Name": val_df["Name"].values,
                  "patient_id": val_df["patient_id"].values,
                  "y_true": vy_swa, "y_pred": vp_swa}
                ).to_csv(OUT_DIR / f"fold{fold}_oof_swa.csv", index=False)

    # Clean up GPU before the next fold.
    del model, swa_model, optimizer, sched, scaler, train_loader, val_loader, bn_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    with open(OUT_DIR / f"fold{fold}_history.json", "w") as f:
        json.dump({"history": history, "best_auc": best_auc, "swa_auc": swa_auc}, f, indent=2)
    return best_auc, swa_auc

In [ ]:
splits = get_splits(df_train, CV, N_SPLITS, SEED)
print(f"CV={CV}, folds={len(splits)}  backbone={BACKBONE}\n")

best_list, swa_list = [], []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    best, swa = train_fold(df_train, tr, va, fold)
    best_list.append(best); swa_list.append(swa)
    print(f"  fold {fold}: best={best:.4f}  swa={swa:.4f}\n")

print(f"Per-fold best AUC: {[f'{a:.4f}' for a in best_list]}  mean {np.mean(best_list):.4f}")
print(f"Per-fold SWA  AUC: {[f'{a:.4f}' for a in swa_list ]}  mean {np.mean(swa_list ):.4f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for hp in sorted(glob.glob(str(OUT_DIR / "fold*_history.json"))):
    h = json.load(open(hp))["history"]
    label = Path(hp).stem.replace("_history", "")
    ax[0].plot([e["epoch"] for e in h], [e["va_loss"] for e in h], marker="o", label=label)
    ax[1].plot([e["epoch"] for e in h], [e["va_auc"]  for e in h], marker="o", label=label)
ax[0].set(title="Validation loss", xlabel="epoch", ylabel="BCE")
ax[1].set(title="Validation AUC",  xlabel="epoch", ylabel="AUC")
ax[1].axhline(0.85, color="red", linestyle="--", alpha=0.5, label="target 0.85")
for a in ax: a.legend(); a.grid(True)
plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# Compare best-epoch OOF vs SWA OOF — usually SWA wins on this kind of small-cohort data.
def report_oof(pattern, label):
    paths = sorted(glob.glob(str(OUT_DIR / pattern)))
    if not paths:
        print(f"({label}: no files matching {pattern})"); return None
    df = pd.concat([pd.read_csv(p) for p in paths])
    print(f"\n=== OOF: {label} ({len(df)} cells, {len(paths)} folds) ===")
    cell_auc = roc_auc_score(df['y_true'], df['y_pred'])
    pp = df.groupby("patient_id").agg(
        mean_pred=("y_pred","mean"), median_pred=("y_pred","median"),
        label=("y_true","first")).sort_values("mean_pred")
    print(pp.to_string())
    pat_auc = roc_auc_score(pp['label'], pp['mean_pred'])
    print(f"cell-level OOF AUC: {cell_auc:.4f}    patient-level AUC: {pat_auc:.4f}")
    return df, cell_auc, pat_auc

_   = report_oof("fold*_oof_best.csv", "best-epoch")
oof_swa = report_oof("fold*_oof_swa.csv",  "SWA")

In [ ]:
def _d4(bf, fl):
    """All 8 D4 transforms (4 rotations × 2 reflections)."""
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    backbone = args.get("backbone", BACKBONE)
    dropout  = float(args.get("dropout", DROPOUT))
    is_swa   = bool(args.get("swa", False))
    base = MultimodalClassifier(pretrained=False, backbone=backbone, dropout=dropout)
    if is_swa:
        model = AveragedModel(base).to(DEVICE)
    else:
        model = base.to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict_one_ckpt(ckpt_path, loader, tta=True):
    model = load_model_from_ckpt(ckpt_path)
    preds = []
    n_aug = 8 if tta else 1
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in (_d4(bf, fl) if tta else [(bf, fl)]):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# Ensemble: best-epoch + SWA across all folds. Each ckpt gets 8-way TTA.
ckpts = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt"))) + \
        sorted(glob.glob(str(OUT_DIR / "fold*_swa.pt")))
print("Ensembling checkpoints:")
for c in ckpts: print("  -", c)

all_preds = []
for c in ckpts:
    t0 = time.time()
    all_preds.append(predict_one_ckpt(c, test_loader, tta=True))
    print(f"  done {Path(c).name} in {time.time()-t0:.1f}s")
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print(f"\nWrote submission.csv  (mean pred = {preds.mean():.3f})")
print(sub.head())
!head /kaggle/working/submission.csv
!wc -l /kaggle/working/submission.csv